In [1]:
# import bibliotek i danych

import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet
from xgboost import XGBRegressor
import lightgbm as lgb

import joblib
import os

df = pd.read_csv(
    "english_bg3_reviews_with_sentiment_best_model.csv",
    sep=";",
    encoding="utf-8"
)

df.head()


Importing plotly failed. Interactive plots will not work.


,review_id,review,voted_up,author_steamid,date_str,review_length,review_word_count,clean_review,is_spam_label,num_emojis,low_quality,is_spam,tokens,sent_flair,review_category,future_expectations,has_expectation
0,209490463,The greatest game of all time. It simply is. \r\n,True,76561198842852247,2025-11-17 20:17:24,43,9,the greatest game of all time it simply is,0,0,0,0,"['the', 'greatest', 'game', 'all', 'time', 'si...",1,Unspecified,0,False
1,209488275,The dude trying to bite me at night was messed...,True,76561198335095296,2025-11-17 19:45:37,55,12,the dude trying to bite me at night was messed...,0,0,0,0,"['the', 'dude', 'trying', 'bite', 'night', 'wa...",0,Unspecified,0,False
2,209488128,"This is probably a great solo game, but my hop...",False,76561198031957731,2025-11-17 19:43:28,131,29,this is probably a great solo game but my hope...,0,0,0,0,"['this', 'probably', 'great', 'solo', 'game', ...",0,"balans, mechanika, soundtrack",1,True
3,209487979,One of the best RPG's of the last decade,True,76561198140742800,2025-11-17 19:41:16,39,9,one of the best rpgs of the last decade,0,0,0,0,"['one', 'the', 'best', 'rpgs', 'the', 'last', ...",1,Unspecified,0,False
4,209482559,Definitely one of the best games I've played i...,True,76561198024036703,2025-11-17 18:19:29,61,12,definitely one of the best games ive played in...,0,0,0,0,"['definitely', 'one', 'the', 'best', 'games', ...",1,Unspecified,0,False


In [2]:
# date_str -> datetime
df["date_str"] = pd.to_datetime(df["date_str"])

# agregacja dzienna na podstawie voted_up (0/1)
daily_ratio = df.groupby(df["date_str"].dt.date)["voted_up"].agg(
    pos=lambda x: (x == 1).sum(),
    neg=lambda x: (x == 0).sum()
)

daily_ratio.index = pd.to_datetime(daily_ratio.index)
daily_ratio.index.name = "date"

# łączna liczba recenzji danego dnia
daily_ratio["total"] = daily_ratio["pos"] + daily_ratio["neg"]

# wyrzucamy dni bez recenzji, żeby ograniczyć ryzyko wystąpienia błędu
daily_ratio = daily_ratio[daily_ratio["total"] > 0]

# dzienny udział pozytywnych recenzji
daily_ratio["positive_ratio_true"] = (
    daily_ratio["pos"] / daily_ratio["total"]
)

# wygładzenie 7-dniowe (target jest stabilniejszy)
daily_ratio["positive_ratio_true_7d"] = (
    daily_ratio["positive_ratio_true"]
    .rolling(7, min_periods=3)
    .mean()
)

daily_ratio.head()


,pos,neg,total,positive_ratio_true,positive_ratio_true_7d
date,,,,,
2024-03-24,64,4,68,0.941176,NaN
2024-03-25,174,5,179,0.972067,NaN
2024-03-26,147,7,154,0.954545,0.955930
2024-03-27,145,18,163,0.889571,0.939340
2024-03-28,98,9,107,0.915888,0.934649


In [3]:
#tylko to, co potrzebne do prognozy
ratio_df = daily_ratio.copy()

# check czy sort po dacie jest odpowiedni
ratio_df = ratio_df.sort_index()

# target: wygładzony 7-dniowy stosunek pozytywnych
TARGET_COL = "positive_ratio_true_7d"

ts = ratio_df[[TARGET_COL]].dropna()
ts.rename(columns={TARGET_COL: "ratio"}, inplace=True)

ts.head()


,ratio
date,
2024-03-26,0.955930
2024-03-27,0.939340
2024-03-28,0.934649
2024-03-29,0.939276
2024-03-30,0.942955


In [4]:
# cechy czasowe + opóźnienia

ts["t"] = np.arange(len(ts))

ts["day_of_week"]  = ts.index.dayofweek
ts["week_of_year"] = ts.index.isocalendar().week.astype(int)
ts["month"]        = ts.index.month

ts["lag1"]     = ts["ratio"].shift(1)
ts["lag7"]     = ts["ratio"].shift(7)
ts["rolling7"] = ts["ratio"].rolling(7).mean()

ts = ts.dropna()

ts.head()


,ratio,t,day_of_week,week_of_year,month,lag1,lag7,rolling7
date,,,,,,,,
2024-04-02,0.939507,7,1,14,4,0.941863,0.955930,0.940279
2024-04-03,0.953839,8,2,14,4,0.939507,0.939340,0.942351
2024-04-04,0.959853,9,3,14,4,0.953839,0.934649,0.945951
2024-04-05,0.957027,10,4,14,4,0.959853,0.939276,0.948487
2024-04-06,0.956069,11,5,14,4,0.957027,0.942955,0.950361


In [5]:
FEATURES = ["t", "day_of_week", "week_of_year", "month", "lag1", "lag7", "rolling7"]

X = ts[FEATURES]
y = ts["ratio"]

train_size = int(len(ts) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

train = ts.iloc[:train_size]
test  = ts.iloc[train_size:]

len(train), len(test)


(476, 119)

In [6]:
results = {}
predictions = {}


In [7]:
# Holt-Winters (Exponential Smoothing)
hw_model = ExponentialSmoothing(
    train["ratio"],
    trend="add",
    seasonal=None,
    initialization_method="estimated"
).fit()

hw_pred = hw_model.forecast(len(test))

results["holt_winters"] = {
    "rmse": np.sqrt(mean_squared_error(test["ratio"], hw_pred)),
    "mae":  mean_absolute_error(test["ratio"], hw_pred)
}
predictions["holt_winters"] = hw_pred


c:\Users\huber\Desktop\inyznierka\python\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


In [8]:
# Prophet

prophet_df = ts[["ratio"]].reset_index().rename(columns={
    "date": "ds",
    "ratio": "y"
})

prophet_train = prophet_df.iloc[:train_size]

prophet_model = Prophet()
prophet_model.fit(prophet_train)

prophet_future = prophet_model.make_future_dataframe(periods=len(test), freq="D")
prophet_forecast = prophet_model.predict(prophet_future)

prophet_pred = prophet_forecast["yhat"].iloc[-len(test):].values

results["prophet"] = {
    "rmse": np.sqrt(mean_squared_error(test["ratio"].values, prophet_pred)),
    "mae":  mean_absolute_error(test["ratio"].values, prophet_pred)
}
predictions["prophet"] = prophet_pred


06:41:37 - cmdstanpy - INFO - Chain [1] start processing
06:41:37 - cmdstanpy - INFO - Chain [1] done processing


In [9]:
# XGBoost

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

results["xgboost"] = {
    "rmse": np.sqrt(mean_squared_error(y_test, xgb_pred)),
    "mae":  mean_absolute_error(y_test, xgb_pred)
}
predictions["xgboost"] = xgb_pred


In [10]:
# LightGBM

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)

results["lightgbm"] = {
    "rmse": np.sqrt(mean_squared_error(y_test, lgb_pred)),
    "mae":  mean_absolute_error(y_test, lgb_pred)
}
predictions["lightgbm"] = lgb_pred


In [11]:
print("PORÓWNANIE MODELI (target: dzienny positive_ratio_true_7d):\n")

for model_name, metric in results.items():
    print(
        f"{model_name:12s} | RMSE: {metric['rmse']:.5f} | MAE: {metric['mae']:.5f}"
    )

best_model = min(results, key=lambda m: results[m]["rmse"])
print("\nNajlepszy model to ===", best_model)


PORÓWNANIE MODELI (target: dzienny positive_ratio_true_7d):

holt_winters | RMSE: 0.03086 | MAE: 0.02663
prophet      | RMSE: 0.01789 | MAE: 0.01476
xgboost      | RMSE: 0.01334 | MAE: 0.00974
lightgbm     | RMSE: 0.01204 | MAE: 0.00890

Najlepszy model to === lightgbm


In [12]:
FUTURE_DAYS = 120

last_date = ts.index.max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1),
                             periods=FUTURE_DAYS,
                             freq="D")

# cechy przyszłości dla modeli ML
future_features = pd.DataFrame(index=future_dates)
future_features["t"] = np.arange(
    ts["t"].max() + 1,
    ts["t"].max() + 1 + FUTURE_DAYS
)
future_features["day_of_week"]  = future_features.index.dayofweek
future_features["week_of_year"] = future_features.index.isocalendar().week.astype(int)
future_features["month"]        = future_features.index.month

# dla lagów używamy ostatnich znanych wartości (proste przybliżenie)
last_row = ts.iloc[-1]
future_features["lag1"]     = last_row["lag1"]
future_features["lag7"]     = last_row["lag7"]
future_features["rolling7"] = last_row["rolling7"]

# predykcja zależnie od najlepszego modelu
if best_model == "xgboost":
    future_pred = xgb_model.predict(future_features[FEATURES])
elif best_model == "lightgbm":
    future_pred = lgb_model.predict(future_features[FEATURES])
elif best_model == "holt_winters":
    future_pred = hw_model.forecast(FUTURE_DAYS)
elif best_model == "prophet":
    prophet_future_all = prophet_model.make_future_dataframe(periods=FUTURE_DAYS, freq="D")
    prophet_forecast_all = prophet_model.predict(prophet_future_all)
    future_part = prophet_forecast_all.tail(FUTURE_DAYS)
    future_pred = future_part["yhat"].values
    future_dates = future_part["ds"].values  # daty z Propheta

# ratio powinno być w [0, 1]
future_pred = np.clip(future_pred, 0.0, 1.0)

forecast_df = pd.DataFrame({
    "date": future_dates,
    "predicted_positive_ratio": future_pred
})

forecast_df.to_csv(
    "english_bg3_positive_ratio_forecast.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

forecast_df.head()


,date,predicted_positive_ratio
0,2025-11-18,0.952951
1,2025-11-19,0.952586
2,2025-11-20,0.952442
3,2025-11-21,0.954911
4,2025-11-22,0.955594


In [13]:
# zapis wytrenowanego modelu

model_path = 'models/ratio_forecast_model.pkl'

if best_model == "prophet":
    joblib.dump(prophet_model, model_path)
elif best_model == "xgboost":
    joblib.dump(xgb_model, model_path)
elif best_model == "lightgbm":
    joblib.dump(lgb_model, model_path)
elif best_model == "holt_winters":
    joblib.dump(hw_model, model_path)